In [ ]:
!pip install transformers datasets torch pandas evaluate nltk
!pip install rouge-score

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration, AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import evaluate
import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename

In [ ]:
# Load datasets
sarcastic_data = pd.read_csv("Final_sarcastic_data.csv")
non_sarcastic_data = pd.read_csv("Final_non_sarcastic_data.csv")


In [ ]:
data = pd.merge(sarcastic_data, non_sarcastic_data, on='id')

data['sarcastic'] = data['sarcastic'].astype(str)
data['non_sarcastic'] = data['non_sarcastic'].astype(str)

# Split data (focus on validation dataset which is 20% of total)
train_data, val_test_data = train_test_split(data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(val_test_data, test_size=0.5, random_state=42)


In [ ]:
def format_data(dataframe):
    return {
        'input_text': dataframe['sarcastic'].tolist(),
        'target_text': dataframe['non_sarcastic'].tolist(),
        'id': dataframe['id'].tolist()
    }

train_dataset = format_data(train_data)
val_dataset = format_data(val_data)
test_dataset = format_data(test_data)

# Model and Tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:

def tokenize_data(inputs, targets, tokenizer, max_len=128):
    input_encodings = tokenizer(
        inputs, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt"
    )
    target_encodings = tokenizer(
        targets, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt"
    )
    return input_encodings, target_encodings

train_encodings, train_labels = tokenize_data(
    train_dataset['input_text'], train_dataset['target_text'], tokenizer
)
val_encodings, val_labels = tokenize_data(
    val_dataset['input_text'], val_dataset['target_text'], tokenizer
)

In [ ]:


class SarcasmDataset(Dataset):
    def __init__(self, encodings, labels, ids=None):
        self.input_ids = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels = labels['input_ids']
        self.ids = ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }
        if self.ids is not None:
            item['id'] = self.ids[idx]
        return item

train_dataset = SarcasmDataset(train_encodings, train_labels)
val_dataset = SarcasmDataset(val_encodings, val_labels, val_dataset['id'])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training (11 epochs as specified)
for epoch in range(11):
    model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

# Save the model and tokenizer
os.makedirs("sarcasm_to_non_sarcasm_model", exist_ok=True)
model.save_pretrained("sarcasm_to_non_sarcasm_model")
tokenizer.save_pretrained("sarcasm_to_non_sarcasm_model")
print("Model saved successfully!")

# Evaluation metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def generate_non_sarcastic_texts(val_dataset, tokenizer, model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    generated_texts = []
    true_texts = []

    with torch.no_grad():
        for item in val_dataset:
            input_ids = item['input_ids'].unsqueeze(0).to(device)
            attention_mask = item['attention_mask'].unsqueeze(0).to(device)

            # Generate the model's output
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

            # Decode the generated text
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Get the true text from the original dataset
            true_text = item.get('original_target', tokenizer.decode(item['labels'], skip_special_tokens=True))

            # Append to results
            generated_texts.append(generated_text)
            true_texts.append(true_text)

    # Calculate ROUGE and BLEU scores
    rouge_scores = rouge.compute(predictions=generated_texts, references=true_texts)
    bleu_scores = bleu.compute(predictions=generated_texts, references=true_texts)

    print("ROUGE Scores:", rouge_scores)
    print("BLEU Scores:", bleu_scores)

    # Prepare output with IDs
    output_texts = [
        {
            'id': item.get('id', 'N/A'),
            'non_sarcastic_text': gen_text
        }
        for item, gen_text in zip(val_dataset, generated_texts)
    ]

    return output_texts

# Generate non-sarcastic texts
generated_val_texts = generate_non_sarcastic_texts(val_dataset, tokenizer, model)

# Save to CSV
output_df = pd.DataFrame(generated_val_texts)
output_df.to_csv("generated_non_sarcastic_texts.csv", index=False)
print("Generated texts saved to generated_non_sarcastic_texts.csv")

# Optional: Print a few examples
print("\nSample Generated Non-Sarcastic Texts:")
print(output_df.head())

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 10: 100%|██████████| 92/92 [00:46<00:00,  1.98it/s, loss=0.202]


Model saved successfully!


ROUGE Scores: {'rouge1': 0.464052481719299, 'rouge2': 0.28012770744416027, 'rougeL': 0.45207290709570147, 'rougeLsum': 0.4525014291080191}
BLEU Scores: {'bleu': 0.1885114622876481, 'precisions': [0.5036231884057971, 0.25237449118046135, 0.15479876160990713, 0.08828828828828829], 'brevity_penalty': 0.9233839551241116, 'length_ratio': 0.9261744966442953, 'translation_length': 828, 'reference_length': 894}
Generated texts saved to generated_non_sarcastic_texts.csv

Sample Generated Non-Sarcastic Texts:
    id                                 non_sarcastic_text
0  811         We need to schedule meetings for the week.
1  441         Another obstacle in my way is frustrating.
2  894         Let’s leave everything to the last minute.
3  328  The meeting is rescheduled for a time I can t ...
4   68  My credit card was declined in front of everyone.


In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load the trained model and tokenizer
model_path = "sarcasm_to_non_sarcasm_model"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load the input sarcastic texts
input_df = pd.read_csv("Sarcastic_IAC_data.csv")

# Function to generate non-sarcastic text
def generate_non_sarcastic_text(text, tokenizer, model, max_length=128):
    # Prepare input
    input_ids = tokenizer.encode(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True
    ).to(device)

    # Generate output
    outputs = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    # Decode and return generated text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate non-sarcastic texts
results = []
for index, row in input_df.iterrows():
    sarcastic_text = row['text']
    non_sarcastic_text = generate_non_sarcastic_text(sarcastic_text, tokenizer, model)

    results.append({
        'id': row['id'],
        'sarcastic_text': sarcastic_text,
        'non_sarcastic_text': non_sarcastic_text
    })

# Convert to DataFrame and save
output_df = pd.DataFrame(results)
output_df.to_csv("IAC_Output.csv", index=False)

print(f"Generated {len(output_df)} non-sarcastic texts.")
print("Output saved to IAC_Output.csv")

# Optional: Print a few samples
print("\nSample Generated Texts:")
print(output_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'Sarcastic_IAC_data.csv'

## Accuracy Finding


In [ ]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load the English Sarcasm Detector model
model_name = "helinivan/english-sarcasm-detector"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Function to preprocess text
def preprocess_data(text):
    """Lowercase and remove punctuation from the text."""
    return text.lower().strip()

# Function to classify text
def classify_sarcasm(text, tokenizer, model):
    """Classify whether a given text is sarcastic or not."""
    inputs = tokenizer(
        [preprocess_data(text)],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1).tolist()[0]
    confidence = max(probs)
    label = probs.index(confidence)
    return {"label": label, "confidence": confidence}

# Load the dataset
file_path = "/content/IAC_Output_t5_small.csv"  # Path to the CSV file
data = pd.read_csv(file_path)

# Check the column name (assumes the column to analyze is named 'non_sarcastic_text')
column_to_analyze = "non_sarcastic_text"
if column_to_analyze not in data.columns:
    raise ValueError(f"Column '{column_to_analyze}' not found in the dataset!")

data = data.reset_index(drop=True)

texts = data[column_to_analyze].dropna().tolist()  # Ensure no NaN values
results = [classify_sarcasm(text, tokenizer, model) for text in texts]

data = data[data[column_to_analyze].notna()]

# Count non-sarcastic sentences
non_sarcastic_count = sum(1 for result in results if result["label"] == 0)
total_sentences = len(texts)

# Calculate accuracy
accuracy = (non_sarcastic_count / total_sentences) * 100

# Output results
print(f"Total Sentences: {total_sentences}")
print(f"Non-Sarcastic Sentences: {non_sarcastic_count}")
print(f"Accuracy (Percentage of Non-Sarcastic Sentences): {accuracy:.2f}%")

# Optional: Save results with predictions
data["sarcasm_label"] = ["Non-Sarcastic" if res["label"] == 0 else "Sarcastic" for res in results]
data["confidence"] = [res["confidence"] for res in results]
data.to_csv("sarcasm_detection_results_iacsmall.csv", index=False)
print("Results saved to 'sarcasm_detection_results_iacsmall.csv'.")


Total Sentences: 834
Non-Sarcastic Sentences: 822
Accuracy (Percentage of Non-Sarcastic Sentences): 98.56%
Results saved to 'sarcasm_detection_results_iacsmall.csv'.


<ipython-input-13-4d48f8d08ebb>:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["sarcasm_label"] = ["Non-Sarcastic" if res["label"] == 0 else "Sarcastic" for res in results]
<ipython-input-13-4d48f8d08ebb>:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["confidence"] = [res["confidence"] for res in results]


In [ ]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load the English Sarcasm Detector model
model_name = "helinivan/english-sarcasm-detector"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Function to preprocess text
def preprocess_data(text):
    """Lowercase and remove punctuation from the text."""
    return text.lower().strip()

# Function to classify text
def classify_sarcasm(text, tokenizer, model):
    """Classify whether a given text is sarcastic or not."""
    inputs = tokenizer(
        [preprocess_data(text)],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1).tolist()[0]
    confidence = max(probs)
    label = probs.index(confidence)
    return {"label": label, "confidence": confidence}

# Load the dataset
file_path = "/content/Bart_Iac_data_predictions.csv"  # Path to the CSV file
data = pd.read_csv(file_path)

# Check the column name (assumes the column to analyze is named 'non_sarcastic_text')
column_to_analyze = "generated"
if column_to_analyze not in data.columns:
    raise ValueError(f"Column '{column_to_analyze}' not found in the dataset!")

data = data.reset_index(drop=True)

texts = data[column_to_analyze].dropna().tolist()  # Ensure no NaN values
results = [classify_sarcasm(text, tokenizer, model) for text in texts]

data = data[data[column_to_analyze].notna()]

# Count non-sarcastic sentences
non_sarcastic_count = sum(1 for result in results if result["label"] == 0)
total_sentences = len(texts)

# Calculate accuracy
accuracy = (non_sarcastic_count / total_sentences) * 100

# Output results
print(f"Total Sentences: {total_sentences}")
print(f"Non-Sarcastic Sentences: {non_sarcastic_count}")
print(f"Accuracy (Percentage of Non-Sarcastic Sentences): {accuracy:.2f}%")

# Optional: Save results with predictions
data["sarcasm_label"] = ["Non-Sarcastic" if res["label"] == 0 else "Sarcastic" for res in results]
data["confidence"] = [res["confidence"] for res in results]
data.to_csv("sarcasm_detection_results_iacbart.csv", index=False)
print("Results saved to 'sarcasm_detection_results_iacbart.csv'.")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Total Sentences: 1630
Non-Sarcastic Sentences: 1628
Accuracy (Percentage of Non-Sarcastic Sentences): 99.88%
Results saved to 'sarcasm_detection_results_iacbart.csv'.
